<div dir="rtl" style="text-align: center; padding: 30px; background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%); color: white; border-radius: 15px;">

# 📊 מדדי ביצוע
## Evaluation Metrics

### מחברת 7 | קורס מדעי הנתונים

**משך:** 3-4 שעות

---

**מה נלמד?**
- למה Accuracy לא מספיק?
- Confusion Matrix - מטריצת הבלבול
- Precision, Recall, F1-Score
- מתי להשתמש בכל מדד?
- דוגמאות מהעולם האמיתי

</div>

<div dir="rtl" style="text-align: right;">

# חלק 1: למה Accuracy לא מספיק? 🤔

---

</div>

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (accuracy_score, confusion_matrix, 
                             precision_score, recall_score, f1_score,
                             classification_report)

plt.style.use('seaborn-v0_8-whitegrid')
np.random.seed(42)

print('Libraries imported successfully!')

<div dir="rtl" style="text-align: right; background-color: #ffebee; padding: 20px; border-radius: 10px; border-right: 4px solid #f44336;">

## ⚠️ בעיית הנתונים הלא מאוזנים

### דוגמה: זיהוי מחלה נדירה

נניח שיש לנו 1000 בדיקות:
- **990** אנשים בריאים 😊
- **10** אנשים חולים 🤒

מודל "טיפש" שתמיד אומר "בריא" יקבל:

$$Accuracy = \frac{990}{1000} = 99\%$$

**99% דיוק!** נשמע מעולה, נכון? 🎉

**אבל רגע...** המודל הזה לא מזהה **אף חולה**! 😱

</div>

In [ ]:
# Demonstration of the problem
# Real labels: 990 healthy, 10 sick
y_true = ['healthy'] * 990 + ['sick'] * 10

# "Dumb" model: always predicts healthy
y_pred_dumb = ['healthy'] * 1000

# Calculate accuracy
accuracy = accuracy_score(y_true, y_pred_dumb)
print(f'"Dumb" Model Accuracy: {accuracy:.1%}')
print(f'\nBut how many sick people did it find?')
print(f'Sick correctly identified: 0 out of 10')
print(f'\n❌ This model is USELESS for detecting disease!')

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-right: 4px solid #4caf50;">

### 💡 המסקנה

**Accuracy לבד לא מספיק!**

צריך מדדים נוספים שמתייחסים לסוגי השגיאות השונים.

</div>

<div dir="rtl" style="text-align: right;">

# חלק 2: מטריצת הבלבול (Confusion Matrix) 📋

---

מטריצה שמראה את **כל סוגי התוצאות** של המודל.

</div>

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px; border-right: 4px solid #2196f3;">

## 📊 ארבע התוצאות האפשריות

נניח שאנחנו מחפשים **חולים** (Positive = חולה, Negative = בריא):

| | חזה: חולה | חזה: בריא |
|---|-----------|----------|
| **באמת: חולה** | ✅ True Positive (TP) | ❌ False Negative (FN) |
| **באמת: בריא** | ❌ False Positive (FP) | ✅ True Negative (TN) |

### פירוש:
- **TP (True Positive):** חולה שזוהה נכון כחולה ✅
- **TN (True Negative):** בריא שזוהה נכון כבריא ✅
- **FP (False Positive):** בריא שזוהה בטעות כחולה ❌ ("אזעקת שווא")
- **FN (False Negative):** חולה שזוהה בטעות כבריא ❌ ("פספוס")

</div>

In [ ]:
# Create a more realistic example
# Medical test with 200 patients
np.random.seed(42)

# Real labels: 160 healthy, 40 sick
y_true = np.array(['healthy'] * 160 + ['sick'] * 40)

# Model predictions (not perfect)
y_pred = np.array(
    ['healthy'] * 150 + ['sick'] * 10 +  # 150 TN, 10 FP
    ['healthy'] * 8 + ['sick'] * 32       # 8 FN, 32 TP
)

print(f'Total patients: {len(y_true)}')
print(f'Actually healthy: {sum(y_true == "healthy")}')
print(f'Actually sick: {sum(y_true == "sick")}')

In [ ]:
# Create and visualize confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=['sick', 'healthy'])

fig, ax = plt.subplots(figsize=(8, 6))

# Custom colors
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Predicted: Sick', 'Predicted: Healthy'],
            yticklabels=['Actually: Sick', 'Actually: Healthy'],
            annot_kws={'size': 20}, ax=ax)

# Add labels
ax.text(0.5, 0.35, 'TP', ha='center', fontsize=12, color='green', fontweight='bold', transform=ax.transData)
ax.text(1.5, 0.35, 'FN', ha='center', fontsize=12, color='red', fontweight='bold', transform=ax.transData)
ax.text(0.5, 1.35, 'FP', ha='center', fontsize=12, color='red', fontweight='bold', transform=ax.transData)
ax.text(1.5, 1.35, 'TN', ha='center', fontsize=12, color='green', fontweight='bold', transform=ax.transData)

ax.set_title('Confusion Matrix', fontsize=14)
plt.tight_layout()
plt.show()

# Extract values
TP = cm[0, 0]  # sick predicted as sick
FN = cm[0, 1]  # sick predicted as healthy
FP = cm[1, 0]  # healthy predicted as sick
TN = cm[1, 1]  # healthy predicted as healthy

print(f'True Positive (TP): {TP} - Sick correctly identified')
print(f'True Negative (TN): {TN} - Healthy correctly identified')
print(f'False Positive (FP): {FP} - Healthy incorrectly labeled as Sick')
print(f'False Negative (FN): {FN} - Sick incorrectly labeled as Healthy')

<div dir="rtl" style="text-align: right;">

# חלק 3: דיוק (Precision) 🎯

---

</div>

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px; border-right: 4px solid #2196f3;">

## 📐 מהו Precision?

**שאלה:** מתוך כל מי ש**חזינו** שהוא חולה, כמה **באמת** חולים?

$$Precision = \frac{TP}{TP + FP}$$

### בעברית:
> **"כמה אני צודק כשאני אומר כן"**

### דוגמה:
- חזיתי ש-42 אנשים חולים
- מתוכם, 32 באמת חולים (TP)
- ו-10 בריאים בטעות (FP)

$$Precision = \frac{32}{32 + 10} = \frac{32}{42} = 76.2\%$$

</div>

In [ ]:
# Calculate Precision
precision = TP / (TP + FP)
print(f'Precision = TP / (TP + FP)')
print(f'Precision = {TP} / ({TP} + {FP})')
print(f'Precision = {TP} / {TP + FP}')
print(f'Precision = {precision:.1%}')

# Using sklearn
precision_sklearn = precision_score(y_true, y_pred, pos_label='sick')
print(f'\nUsing sklearn: {precision_sklearn:.1%}')

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 15px; border-radius: 10px; border-right: 4px solid #ff9800;">

### 🎯 מתי Precision חשוב?

כש**אזעקות שווא יקרות** (FP):
- **ספאם:** אם אימייל חשוב מסומן כספאם - זה בעייתי
- **מעצר:** אם עוצרים אדם חף מפשע - זה חמור
- **השקעה:** אם קונים מניה על סמך חיזוי שגוי - מפסידים כסף

</div>

<div dir="rtl" style="text-align: right;">

# חלק 4: רגישות (Recall) 🔍

---

</div>

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px; border-right: 4px solid #2196f3;">

## 📐 מהו Recall?

**שאלה:** מתוך כל מי ש**באמת** חולה, כמה **מצאנו**?

$$Recall = \frac{TP}{TP + FN}$$

### בעברית:
> **"כמה מהחולים הצלחתי למצוא"**

### דוגמה:
- יש 40 חולים באמת
- מצאנו 32 מהם (TP)
- פספסנו 8 (FN)

$$Recall = \frac{32}{32 + 8} = \frac{32}{40} = 80\%$$

</div>

In [ ]:
# Calculate Recall
recall = TP / (TP + FN)
print(f'Recall = TP / (TP + FN)')
print(f'Recall = {TP} / ({TP} + {FN})')
print(f'Recall = {TP} / {TP + FN}')
print(f'Recall = {recall:.1%}')

# Using sklearn
recall_sklearn = recall_score(y_true, y_pred, pos_label='sick')
print(f'\nUsing sklearn: {recall_sklearn:.1%}')

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 15px; border-radius: 10px; border-right: 4px solid #ff9800;">

### 🔍 מתי Recall חשוב?

כש**פספוסים יקרים** (FN):
- **סרטן:** לא לזהות חולה = אסון
- **הונאה:** לא לזהות הונאה = הפסד כספי
- **ביטחון:** לא לזהות איום = סכנה

</div>

<div dir="rtl" style="text-align: right;">

## 🎨 ויזואליזציה: Precision vs Recall

</div>

In [ ]:
# Visual explanation
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Precision visualization
ax1 = axes[0]
predicted_positive = TP + FP
ax1.barh(['Predicted Sick'], [predicted_positive], color='lightblue', edgecolor='black')
ax1.barh(['Predicted Sick'], [TP], color='green', edgecolor='black', label=f'TP = {TP}')
ax1.set_xlim(0, 50)
ax1.set_title(f'Precision = TP / (TP + FP) = {TP}/{predicted_positive} = {precision:.1%}', fontsize=12)
ax1.set_xlabel('Number of Patients')
ax1.legend()
ax1.text(TP + 2, 0, f'FP = {FP}', va='center', fontsize=11)

# Recall visualization
ax2 = axes[1]
actually_positive = TP + FN
ax2.barh(['Actually Sick'], [actually_positive], color='lightyellow', edgecolor='black')
ax2.barh(['Actually Sick'], [TP], color='green', edgecolor='black', label=f'TP = {TP}')
ax2.set_xlim(0, 50)
ax2.set_title(f'Recall = TP / (TP + FN) = {TP}/{actually_positive} = {recall:.1%}', fontsize=12)
ax2.set_xlabel('Number of Patients')
ax2.legend()
ax2.text(TP + 2, 0, f'FN = {FN}', va='center', fontsize=11)

plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right;">

# חלק 5: F1-Score - האיזון ⚖️

---

</div>

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px; border-right: 4px solid #2196f3;">

## 📐 מהו F1-Score?

**ממוצע הרמוני** של Precision ו-Recall:

$$F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}$$

### למה ממוצע הרמוני ולא רגיל?

כי הוא "מעניש" ערכים קיצוניים:
- אם Precision=100% ו-Recall=0% → F1=0% (לא 50%!)
- רק אם **שניהם** גבוהים, F1 יהיה גבוה

</div>

In [ ]:
# Calculate F1-Score
f1 = 2 * (precision * recall) / (precision + recall)
print(f'F1 = 2 × (Precision × Recall) / (Precision + Recall)')
print(f'F1 = 2 × ({precision:.3f} × {recall:.3f}) / ({precision:.3f} + {recall:.3f})')
print(f'F1 = {f1:.1%}')

# Using sklearn
f1_sklearn = f1_score(y_true, y_pred, pos_label='sick')
print(f'\nUsing sklearn: {f1_sklearn:.1%}')

In [ ]:
# Demonstrate harmonic mean vs arithmetic mean
print('Why Harmonic Mean?')
print('=' * 40)

# Extreme case
p_extreme = 1.0  # 100%
r_extreme = 0.0  # 0%

arithmetic = (p_extreme + r_extreme) / 2
# Harmonic mean with 0 would be undefined, so use small value
r_small = 0.01
harmonic = 2 * (p_extreme * r_small) / (p_extreme + r_small)

print(f'Precision = 100%, Recall ≈ 0%')
print(f'Arithmetic Mean: {arithmetic:.1%} (misleading!)')
print(f'Harmonic Mean (F1): {harmonic:.1%} (realistic!)')
print(f'\n💡 Harmonic mean "punishes" extreme imbalance')

<div dir="rtl" style="text-align: right;">

# חלק 6: סיכום המדדים 📋

---

</div>

In [ ]:
# Summary of all metrics
accuracy = accuracy_score(y_true, y_pred)

print('SUMMARY OF METRICS')
print('=' * 50)
print(f'Accuracy:  {accuracy:.1%}  (Overall correctness)')
print(f'Precision: {precision:.1%}  (When I say sick, am I right?)')
print(f'Recall:    {recall:.1%}  (Did I find all sick patients?)')
print(f'F1-Score:  {f1:.1%}  (Balance of Precision & Recall)')

In [ ]:
# Visual comparison
metrics = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
values = [accuracy, precision, recall, f1]
colors = ['#3498db', '#2ecc71', '#e74c3c', '#9b59b6']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(metrics, values, color=colors, edgecolor='black')

# Add value labels
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, 
            f'{val:.1%}', ha='center', fontsize=12, fontweight='bold')

ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Comparison of Evaluation Metrics', fontsize=14)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

<div dir="rtl" style="text-align: right; background-color: #fff8e1; padding: 20px; border-radius: 10px;">

## 📋 טבלת סיכום - מתי להשתמש בכל מדד?

| מדד | נוסחה | שאלה שעונה | מתי חשוב? |
|-----|-------|-----------|----------|
| **Accuracy** | (TP+TN) / הכל | כמה צדקתי בסה"כ? | נתונים מאוזנים |
| **Precision** | TP / (TP+FP) | כשאני אומר "כן", כמה אני צודק? | אזעקות שווא יקרות |
| **Recall** | TP / (TP+FN) | כמה "כן" אמיתיים מצאתי? | פספוסים יקרים |
| **F1-Score** | ממוצע הרמוני | איזון כללי | כשצריך שניהם |

</div>

<div dir="rtl" style="text-align: right;">

# חלק 7: דוגמאות מהעולם האמיתי 🌍

---

</div>

<div dir="rtl" style="text-align: right; background-color: #ffebee; padding: 20px; border-radius: 10px; border-right: 4px solid #f44336;">

## 🏥 דוגמה 1: בדיקת סרטן

**מה יותר גרוע?**
- **FP (אזעקת שווא):** אדם בריא נשלח לבדיקות נוספות (מלחיץ, אבל לא מסוכן)
- **FN (פספוס):** חולה סרטן לא מזוהה ← **סכנת חיים!**

**לכן:** נעדיף **Recall גבוה** (גם אם זה אומר יותר אזעקות שווא)

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 20px; border-radius: 10px; border-right: 4px solid #4caf50;">

## 📧 דוגמה 2: סינון ספאם

**מה יותר גרוע?**
- **FP (אזעקת שווא):** אימייל חשוב מסומן כספאם ← **מפספסים הודעה חשובה!**
- **FN (פספוס):** ספאם נכנס לתיבה (מעצבן, אבל לא נורא)

**לכן:** נעדיף **Precision גבוה** (עדיף קצת ספאם מאשר לאבד אימייל חשוב)

</div>

<div dir="rtl" style="text-align: right; background-color: #fff3e0; padding: 20px; border-radius: 10px; border-right: 4px solid #ff9800;">

## 💳 דוגמה 3: זיהוי הונאות בכרטיסי אשראי

**מה יותר גרוע?**
- **FP:** עסקה לגיטימית נחסמת (מבאס ללקוח)
- **FN:** הונאה לא מזוהה (הפסד כספי)

**לכן:** צריך **איזון** - משתמשים ב-**F1-Score**

</div>

<div dir="rtl" style="text-align: right;">

# חלק 8: דוח סיווג מלא 📊

---

sklearn נותן לנו את כל המדדים בפקודה אחת:

</div>

In [ ]:
# Full classification report
print('CLASSIFICATION REPORT')
print('=' * 55)
print(classification_report(y_true, y_pred))

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px; border-right: 4px solid #4caf50;">

### 💡 קריאת הדוח

- **precision, recall, f1-score** - לכל מחלקה בנפרד
- **support** - מספר הדגימות האמיתיות בכל מחלקה
- **accuracy** - דיוק כללי
- **macro avg** - ממוצע פשוט של כל המחלקות
- **weighted avg** - ממוצע משוקלל לפי גודל המחלקה

</div>

<div dir="rtl" style="text-align: right;">

# חלק 9: תרגול 📝

---

</div>

In [ ]:
# Exercise: Calculate metrics for a new scenario
print('EXERCISE: Email Spam Filter')
print('=' * 40)
print('Your model results:')
print('  TP = 80 (spam correctly identified)')
print('  TN = 900 (legitimate email correctly identified)')
print('  FP = 20 (legitimate email marked as spam)')
print('  FN = 50 (spam not caught)')
print('\nCalculate:')
print('1. Accuracy')
print('2. Precision')
print('3. Recall')
print('4. F1-Score')
print('5. Which metric is most important for spam filtering?')

In [ ]:
# Solution (run after attempting)
TP_ex, TN_ex, FP_ex, FN_ex = 80, 900, 20, 50

accuracy_ex = (TP_ex + TN_ex) / (TP_ex + TN_ex + FP_ex + FN_ex)
precision_ex = TP_ex / (TP_ex + FP_ex)
recall_ex = TP_ex / (TP_ex + FN_ex)
f1_ex = 2 * (precision_ex * recall_ex) / (precision_ex + recall_ex)

print('SOLUTION')
print('=' * 40)
print(f'1. Accuracy:  {accuracy_ex:.1%}')
print(f'2. Precision: {precision_ex:.1%}')
print(f'3. Recall:    {recall_ex:.1%}')
print(f'4. F1-Score:  {f1_ex:.1%}')
print(f'\n5. Precision is most important!')
print(f'   We want to avoid marking legitimate emails as spam (FP).')

<div dir="rtl" style="text-align: right; background-color: #e3f2fd; padding: 20px; border-radius: 10px;">

# 📝 סיכום

## הנוסחאות החשובות:

| מדד | נוסחה |
|-----|-------|
| Accuracy | (TP + TN) / (TP + TN + FP + FN) |
| Precision | TP / (TP + FP) |
| Recall | TP / (TP + FN) |
| F1-Score | 2 × (Precision × Recall) / (Precision + Recall) |

## כלל הזהב:
- **נתונים מאוזנים** → Accuracy מספיק
- **נתונים לא מאוזנים** → חייבים Precision, Recall, F1
- **תמיד להסתכל על Confusion Matrix!**

</div>

<div dir="rtl" style="text-align: right; background-color: #e8f5e9; padding: 15px; border-radius: 10px;">

## 👀 בשיעור הבא...

**מחברת 8: רגרסיה לינארית**

נעבור מסיווג (קטגוריות) לרגרסיה (ערכים רציפים)!

</div>